<a href="https://colab.research.google.com/github/rwcitek/Medicare-data/blob/main/medicare-data-minimal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medicare data



From [Doctors and clinicians datasets](https://data.cms.gov/provider-data/search?theme=Doctors%20and%20clinicians) page.

- [National Downloadable File]( https://data.cms.gov/provider-data/dataset/mj5m-pzi6 )
- [Data Dictionary]( https://data.cms.gov/provider-data/sites/default/files/data_dictionaries/physician/DOC_Data_Dictionary.pdf )


From the National Downloadable File URL, we see that the dataset ID is `mj5m-pzi6`.  We can use that to query the CMS API to get the link to the CSV file.



In [ ]:
%%capture pkg_output
%%bash
apt-get update
apt-get install -y tree jq


In [ ]:
import requests
import pandas as pd


In [ ]:
dataset_id = "mj5m-pzi6"
url = f"https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items/{dataset_id}"

dataset_id, url

In [ ]:
response = requests.get(url)
response


In [ ]:
metadata = response.json()
metadata


In [ ]:
url = metadata["distribution"][0]["downloadURL"]
url


## Download the CSV file


In [ ]:
!curl -O {url}


In [ ]:
csv_file = url.split("/")[-1]
csv_file


In [ ]:
!ls -la {csv_file}

In [ ]:
!wc -l {csv_file}

## DuckDB: query a CSV file



In [ ]:
import duckdb


In [ ]:
duckdb.query(f"""
  SELECT *
  FROM read_csv(
    "{csv_file}",
    types={{'Telephone Number': 'VARCHAR'}},
    auto_detect=True
  )
  LIMIT 10
""").show() ;


## DuckDB: query a CSV file with a CTE


In [ ]:
duckdb.query(f"""
  WITH
  medicare as (
    SELECT *
    FROM read_csv(
      "{csv_file}",
      types={{'Telephone Number': 'VARCHAR'}},
      auto_detect=True
    )
  ),

  medicare_nm as (
    SELECT *
    FROM medicare
    WHERE State = 'NM'
  ),

  medicare_abq as (
    SELECT *
    FROM medicare_nm
    WHERE "City/Town" IN ['ALBUQERQUE', 'RIO RANCHO']
  )

  SELECT *
  FROM medicare_abq
  ORDER BY "City/Town", NPI
  LIMIT 10
""").show()


In [ ]:
query = f"""
  SELECT *
  FROM read_csv(
    "{csv_file}",
    types={{'Telephone Number': 'VARCHAR'}},
    auto_detect=True
  )
""".strip()
print(query)


In [ ]:
rel = duckdb.sql(query)


## Convert CSV to hive-partitioned parquet


In [ ]:
export_query = '''
  COPY rel TO 'medicare_DAC_national'
  (FORMAT PARQUET, PARTITION_BY (State), OVERWRITE_OR_IGNORE 1);
'''
print(export_query)


In [ ]:
duckdb.sql(export_query)


In [ ]:
!ls -la


In [ ]:
!tree medicare_DAC_national/

In [ ]:
!du -ms medicare_DAC_national/


## Verify that duckdb can query parquet files


In [ ]:
dataset_path = "medicare_DAC_national/**/*.parquet"
dataset_path


In [ ]:
query = f"SELECT * FROM read_parquet('{dataset_path}')"
query


In [ ]:
rel = duckdb.sql(query)


In [ ]:
print(f"Total rows found across all partitions: {len(rel):,}")


In [ ]:
rel.show()
